# LeetCode 3385 — Minimum Time to Break Locks II bằng Python

- Link: https://leetcode.com/problems/minimum-time-to-break-locks-ii/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Bài toán phân công** (assignment): khóa ↔ thứ tự phá; **thuật toán Hungary** `O(n³)`, **luồng chi phí nhỏ nhất**; so sánh với DP bitmask của bản I

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề, ví dụ, ràng buộc lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3385/). Mã trên doocs dùng `findMinimumTime(self, a)`; tên tham số trên LeetCode **có thể** là `strength` (như mô tả đề) — notebook dùng `findMinimumTime(self, strength)`. Vui lòng **đối chiếu lại** chữ ký nếu có tài khoản Premium.

Skeleton (suy ra):

```python
class Solution:
    def findMinimumTime(self, strength: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra khi phá khóa thứ `j` (đếm từ 0) thì hệ số là `X = j + 1` và thời gian để năng lượng đạt `s` là `⌈s / (j + 1)⌉` — **chỉ phụ thuộc vị trí** của khóa trong thứ tự.
2. Mô hình thành **bài toán phân công**: ma trận chi phí `C[i][j] = ⌈strength[i] / (j+1)⌉`, chọn hoán vị tổng nhỏ nhất.
3. Cài **thuật toán Hungary** (dạng thế vị `u, v`) `O(n³)` — đủ nhanh với `n ≤ 80`.
4. Hiểu cách mô hình bằng **luồng chi phí nhỏ nhất** (như lời giải doocs).
5. Thấy vì sao **tham lam** (sắp tăng/giảm) và **DP bitmask** (bản I) không dùng được ở đây.

## 2. Đề bài & ví dụ

Có `n` khóa, khóa `i` cần năng lượng `strength[i]`. Thanh kiếm có năng lượng ban đầu 0 và hệ số `X = 1`. Mỗi phút năng lượng tăng thêm `X`. Khi năng lượng ≥ `strength[i]` có thể phá khóa `i`; sau đó năng lượng về 0 và `X` tăng 1. Tìm thời gian ít nhất để phá hết các khóa.

**Ví dụ 1:** `[3,4,1]` → `4` (phá khóa 3, rồi khóa 2, rồi khóa 1: `1 + 2 + 1`).

**Ví dụ 2:** `[2,5,4]` → `6`.

**Ràng buộc (theo doocs):** `1 ≤ n ≤ 80`, `1 ≤ strength[i] ≤ 10^6`.

In [1]:
from typing import List
import itertools

examples = [([3, 4, 1], 4), ([2, 5, 4], 6)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[3, 4, 1] -> kỳ vọng 4
[2, 5, 4] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Thời gian phá một khóa ở vị trí `j`

Hệ số là `j + 1`; sau `m` phút năng lượng là `m (j + 1)` ⇒ cần `m = ⌈s / (j+1)⌉`. Trong Python: `(s + j) // (j + 1)`.

In [2]:
def cost(s, j):
    return (s + j) // (j + 1)

def simulate(strength, order):
    total, X = 0, 1
    for i in order:
        energy, minutes = 0, 0
        while energy < strength[i]:
            energy += X; minutes += 1
        total += minutes; X += 1
    return total

print(simulate([3, 4, 1], [2, 1, 0]), sum(cost([3, 4, 1][i], j) for j, i in enumerate([2, 1, 0])))

4 4


### 3.2. Bài toán phân công

Cho ma trận `C` `n × n`, chọn hoán vị `σ` tối thiểu `Σ C[i][σ(i)]`. Thuật toán Hungary duy trì thế vị hàng `u` và cột `v` với `u_i + v_j ≤ C[i][j]`; mỗi lần thêm một hàng, tìm **đường tăng** ngắn nhất theo chi phí rút gọn `C[i][j] − u_i − v_j` (giống Dijkstra trên đồ thị hai phía). Tổng `O(n³)`.

### 3.3. Vì sao không tham lam được

Chi phí `⌈s/(j+1)⌉` không "tách" được thành tích hai yếu tố như `s · w_j` (khi đó sắp xếp là tối ưu theo bất đẳng thức hoán vị); phép làm tròn lên phá vỡ cấu trúc đó.

In [3]:
s = [3, 4, 1]
print('sắp tăng:', sum(cost(x, j) for j, x in enumerate(sorted(s))), '| sắp giảm:', sum(cost(x, j) for j, x in enumerate(sorted(s, reverse=True))), '| tối ưu: 4')

sắp tăng: 5 | sắp giảm: 7 | tối ưu: 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[3, 4, 1]`: ma trận chi phí (hàng = khóa, cột = vị trí `j = 0, 1, 2`):

| khóa \ vị trí | 0 (X=1) | 1 (X=2) | 2 (X=3) |
|---|---|---|---|
| 3 | 3 | 2 | 1 |
| 4 | 4 | 2 | 2 |
| 1 | 1 | 1 | 1 |

Chọn `khóa 1 → vị trí 0` (1), `khóa 4 → vị trí 1` (2), `khóa 3 → vị trí 2` (1) ⇒ tổng 4 — không trùng thứ tự sắp tăng (`1, 3, 4` cho `1 + 2 + 2 = 5`).

In [4]:
for x in s:
    print(x, [cost(x, j) for j in range(3)])

3 [3, 2, 1]
4 [4, 2, 2]
1 [1, 1, 1]


## 5. Cách 1 — Brute Force: thử mọi hoán vị

In [5]:
def min_time_brute(strength):
    n = len(strength)
    return min(sum(cost(strength[i], j) for j, i in enumerate(p)) for p in itertools.permutations(range(n)))

for a, exp in examples:
    print(min_time_brute(a), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6


**Độ phức tạp:** `O(n! · n)` — chỉ cho `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

Dựng `C[i][j] = ⌈strength[i]/(j+1)⌉` rồi giải bài toán phân công bằng **Hungary** `O(n³) = 5·10^5` phép với `n = 80`.

(Ở bản I — LC 3376, `n ≤ 8` và hệ số tăng `K` — DP bitmask `O(2^n · n)` là đủ; với `n = 80` thì `2^80` là bất khả thi.)

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP bitmask (bản I)

`dp[mask]` = thời gian nhỏ nhất để phá các khóa trong `mask` (theo thứ tự nào đó); khóa tiếp theo ở vị trí `j = popcount(mask)`.

In [6]:
def min_time_bitmask(strength):
    n = len(strength)
    INF = float('inf')
    dp = [INF] * (1 << n)
    dp[0] = 0
    for m in range(1 << n):
        if dp[m] == INF:
            continue
        j = bin(m).count('1')
        for i in range(n):
            if not m >> i & 1:
                v = dp[m] + cost(strength[i], j)
                if v < dp[m | 1 << i]:
                    dp[m | 1 << i] = v
    return dp[-1]

for a, exp in examples:
    print(min_time_bitmask(a), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6


### 7.2. Cách 3 — Thuật toán Hungary (lời giải chính)

In [7]:
def hungarian(C):
    n = len(C)
    INF = float('inf')
    u = [0] * (n + 1); v = [0] * (n + 1)
    p = [0] * (n + 1)                 # p[j] = hàng được gán cho cột j (1-based, 0 = chưa)
    way = [0] * (n + 1)
    for i in range(1, n + 1):
        p[0] = i
        j0 = 0
        minv = [INF] * (n + 1)
        used = [False] * (n + 1)
        while True:
            used[j0] = True
            i0, delta, j1 = p[j0], INF, 0
            for j in range(1, n + 1):
                if not used[j]:
                    cur = C[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j], way[j] = cur, j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
            for j in range(n + 1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:                    # đảo đường tăng
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    assignment = [0] * n
    for j in range(1, n + 1):
        assignment[p[j] - 1] = j - 1
    return -v[0], assignment

def min_time_hungarian(strength):
    n = len(strength)
    return hungarian([[cost(strength[i], j) for j in range(n)] for i in range(n)])[0]

for a, exp in examples:
    print(min_time_hungarian(a), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6


### 7.3. Cách 4 — Luồng chi phí nhỏ nhất (Bellman–Ford/SPFA trên đồ thị dư)

Nguồn → mỗi khóa (sức chứa 1, chi phí 0); khóa `i` → vị trí `j` (1, `C[i][j]`); vị trí → đích (1, 0). Đẩy `n` đơn vị luồng, mỗi lần theo đường đi ngắn nhất trên đồ thị dư (có cạnh ngược chi phí âm ⇒ dùng SPFA). `O(n · n² · …)` — chậm hơn Hungary nhưng tổng quát.

In [8]:
from collections import deque

def min_cost_flow_assignment(C):
    n = len(C)
    S, T = 2 * n, 2 * n + 1
    N = 2 * n + 2
    graph = [[] for _ in range(N)]                    # cạnh: [đích, sức chứa, chi phí, chỉ số cạnh ngược]
    def add(a, b, cap, w):
        graph[a].append([b, cap, w, len(graph[b])])
        graph[b].append([a, 0, -w, len(graph[a]) - 1])
    for i in range(n):
        add(S, i, 1, 0); add(n + i, T, 1, 0)
        for j in range(n):
            add(i, n + j, 1, C[i][j])
    total = 0
    for _ in range(n):
        dist = [float('inf')] * N; inq = [False] * N; prev = [None] * N
        dist[S] = 0; dq = deque([S])
        while dq:
            x = dq.popleft(); inq[x] = False
            for idx, (y, cap, w, _) in enumerate(graph[x]):
                if cap and dist[x] + w < dist[y]:
                    dist[y] = dist[x] + w; prev[y] = (x, idx)
                    if not inq[y]:
                        inq[y] = True; dq.append(y)
        y = T
        while y != S:
            x, idx = prev[y]
            e = graph[x][idx]
            e[1] -= 1
            graph[y][e[3]][1] += 1
            y = x
        total += dist[T]
    return total

def min_time_flow(strength):
    n = len(strength)
    return min_cost_flow_assignment([[cost(strength[i], j) for j in range(n)] for i in range(n)])

for a, exp in examples:
    print(min_time_flow(a), '| kỳ vọng', exp)

4 | kỳ vọng 4
6 | kỳ vọng 6


## 8. TRACE từng bước

In phép gán tối ưu của Hungary cho ví dụ 2 và mô phỏng lại theo thứ tự đó.

In [9]:
def trace(strength):
    n = len(strength)
    C = [[cost(strength[i], j) for j in range(n)] for i in range(n)]
    total, assign = hungarian(C)
    order = sorted(range(n), key=lambda i: assign[i])
    print('ma trận chi phí:')
    for i in range(n):
        print('  khóa', strength[i], C[i])
    for i in order:
        print(f"vị trí {assign[i]} (X = {assign[i] + 1}): phá khóa strength={strength[i]} tốn {C[i][assign[i]]} phút")
    print('tổng:', total, '| mô phỏng:', simulate(strength, order))

trace([2, 5, 4])

ma trận chi phí:
  khóa 2 [2, 1, 1]
  khóa 5 [5, 3, 2]
  khóa 4 [4, 2, 2]
vị trí 0 (X = 1): phá khóa strength=2 tốn 2 phút
vị trí 1 (X = 2): phá khóa strength=4 tốn 2 phút
vị trí 2 (X = 3): phá khóa strength=5 tốn 2 phút
tổng: 6 | mô phỏng: 6


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def findMinimumTime(self, strength: List[int]) -> int:
        n = len(strength)
        C = [[(strength[i] + j) // (j + 1) for j in range(n)] for i in range(n)]
        INF = float('inf')
        u = [0] * (n + 1)
        v = [0] * (n + 1)
        p = [0] * (n + 1)
        way = [0] * (n + 1)
        for i in range(1, n + 1):
            p[0] = i
            j0 = 0
            minv = [INF] * (n + 1)
            used = [False] * (n + 1)
            while True:
                used[j0] = True
                i0 = p[j0]
                delta = INF
                j1 = 0
                row = C[i0 - 1]
                ui = u[i0]
                for j in range(1, n + 1):
                    if not used[j]:
                        cur = row[j - 1] - ui - v[j]
                        if cur < minv[j]:
                            minv[j] = cur
                            way[j] = j0
                        if minv[j] < delta:
                            delta = minv[j]
                            j1 = j
                for j in range(n + 1):
                    if used[j]:
                        u[p[j]] += delta
                        v[j] -= delta
                    else:
                        minv[j] -= delta
                j0 = j1
                if p[j0] == 0:
                    break
            while j0:
                j1 = way[j0]
                p[j0] = p[j1]
                j0 = j1
        return -v[0]

for a, exp in examples:
    got = Solution().findMinimumTime(a)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
6 OK


## 10. Vì sao thuật toán đúng

1. **Mô hình:** thời gian phá khóa `i` chỉ phụ thuộc vị trí `j` của nó (hệ số `j + 1`, năng lượng về 0 sau mỗi lần phá) ⇒ tổng thời gian của một thứ tự = `Σ C[i][σ(i)]` với `σ` là song ánh khóa ↔ vị trí.
2. **Hungary** (định lý đối ngẫu của quy hoạch tuyến tính cho bài phân công): duy trì `u_i + v_j ≤ C[i][j]` và các cặp đã gán có dấu bằng; khi gán đủ `n` cặp, tổng chi phí = `Σ u + Σ v`, là cận dưới của mọi phép gán ⇒ tối ưu. `−v[0]` trong cài đặt chính là giá trị tối ưu.
3. **Luồng chi phí nhỏ nhất** với `n` đơn vị trên đồ thị hai phía sức chứa 1 ⇔ phép gán tối ưu (đồ thị dư không có chu trình âm sau mỗi lần tăng theo đường ngắn nhất).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hoán vị | `O(n! · n)` | `O(n)` |
| 2. DP bitmask | `O(2^n · n)` | `O(2^n)` |
| 3. Hungary | `O(n³)` | `O(n²)` |
| 4. Min-cost flow (SPFA) | `O(n · E · …)`, `E = n²` | `O(n²)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'bitmask': min_time_bitmask, 'hungarian': min_time_hungarian, 'flow': min_time_flow,
         'submit': lambda a: Solution().findMinimumTime(a)}
edge = [([1], 1), ([10 ** 6], 10 ** 6), ([5, 5], 8), ([1, 1, 1], 3), ([6, 1], 4), ([1, 6], 4)]
for a, exp in examples + edge:
    assert min_time_brute(a) == exp, (a, min_time_brute(a))
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (và mô phỏng)

In [12]:
import random
rng = random.Random(3385)
cnt = 0
for _ in range(400):
    n = rng.randint(1, 7)
    a = [rng.randint(1, rng.choice([5, 30, 1000])) for _ in range(n)]
    exp = min_time_brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    total, assign = hungarian([[cost(a[i], j) for j in range(n)] for i in range(n)])
    assert simulate(a, sorted(range(n), key=lambda i: assign[i])) == total
    cnt += 1
for _ in range(50):
    a = [rng.randint(1, 10 ** 6) for _ in range(rng.randint(1, 14))]
    assert min_time_bitmask(a) == Solution().findMinimumTime(a)
print('Khớp vét cạn trên', cnt, 'bộ (kể cả mô phỏng đúng luật đề); khớp bitmask trên 50 bộ n ≤ 14.')

Khớp vét cạn trên 400 bộ (kể cả mô phỏng đúng luật đề); khớp bitmask trên 50 bộ n ≤ 14.


### 12.3. Ca lớn `n = 80`

In [13]:
import time
cases = {
    'ngẫu nhiên lớn': [rng.randint(1, 10 ** 6) for _ in range(80)],
    'toàn 10^6': [10 ** 6] * 80,
    'nhỏ, nhiều hòa': [rng.randint(1, 10) for _ in range(80)],
}
for name, a in cases.items():
    t = time.time(); r = Solution().findMinimumTime(a); t1 = time.time() - t
    t = time.time(); r2 = min_time_flow(a); t2 = time.time() - t
    assert r == r2
    print(f"{name:16s}: {r:9d} | Hungary {t1:.2f}s | min-cost flow {t2:.2f}s")

ngẫu nhiên lớn  :    980377 | Hungary 0.03s | min-cost flow 0.71s
toàn 10^6       :   4965516 | Hungary 0.03s | min-cost flow 0.05s
nhỏ, nhiều hòa  :        80 | Hungary 0.03s | min-cost flow 0.05s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam: phá khóa yếu trước (sắp tăng)

In [14]:
def wrong_sort_asc(strength):
    return sum(cost(x, j) for j, x in enumerate(sorted(strength)))

print('sai :', wrong_sort_asc([3, 4, 1]), '| đúng:', Solution().findMinimumTime([3, 4, 1]))

sai : 5 | đúng: 4


**Vì sao sai:** làm tròn lên khiến lợi ích của hệ số lớn không tỉ lệ với `strength` — ở ví dụ 1, khóa 4 ở vị trí `X = 2` và khóa 3 ở `X = 3` (`2 + 1`) tốt hơn ngược lại (`2 + 2`).

### Lỗi 2 — Tham lam: phá khóa mạnh trước (sắp giảm)

In [15]:
def wrong_sort_desc(strength):
    return sum(cost(x, j) for j, x in enumerate(sorted(strength, reverse=True)))

print('sai :', wrong_sort_desc([3, 4, 1]), '| đúng:', Solution().findMinimumTime([3, 4, 1]))

sai : 7 | đúng: 4


**Vì sao sai:** khóa mạnh nhất phá đầu tiên với `X = 1` tốn nguyên `strength` phút.

### Lỗi 3 — Dùng phép chia **làm tròn xuống**

In [16]:
def wrong_floor(strength):
    n = len(strength)
    return hungarian([[strength[i] // (j + 1) for j in range(n)] for i in range(n)])[0]

print('sai :', wrong_floor([2, 5, 4]), '| đúng:', Solution().findMinimumTime([2, 5, 4]))

sai : 5 | đúng: 6


**Vì sao sai:** năng lượng phải **đạt ít nhất** `strength` ⇒ số phút là `⌈s / X⌉`.

### Lỗi 4 — Tham lam theo từng vị trí: mỗi bước chọn khóa rẻ nhất cho vị trí hiện tại

In [17]:
def wrong_greedy_positions(strength):
    left = list(strength)
    total = 0
    for j in range(len(strength)):
        k = min(range(len(left)), key=lambda t: (cost(left[t], j), left[t]))
        total += cost(left[k], j)
        left.pop(k)
    return total

rng = random.Random(4)
ce = next(a for a in ([rng.randint(1, 20) for _ in range(rng.randint(2, 6))] for _ in range(20000))
          if wrong_greedy_positions(a) != min_time_brute(a))
print('phản ví dụ:', ce, '| sai:', wrong_greedy_positions(ce), '| đúng:', min_time_brute(ce))

phản ví dụ: [6, 4, 9, 7] | sai: 13 | đúng: 12


**Vì sao sai:** chọn rẻ nhất ở vị trí đầu (luôn là khóa yếu nhất) có thể "lãng phí" vị trí có hệ số lớn sau này. Bài toán phân công cần xét toàn cục.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn thứ tự, chi phí mỗi phần tử chỉ phụ thuộc vị trí của nó" ⇒ **bài toán phân công**.
- `n ≤ 20` ⇒ DP bitmask; `n ≤ 500` ⇒ Hungary `O(n³)`; tổng quát ⇒ min-cost flow.
- Nếu chi phí có dạng tích `a_i · w_j` ⇒ sắp xếp (bất đẳng thức hoán vị) là đủ — nhưng làm tròn phá dạng này.
- Luôn kiểm chứng tham lam bằng vét cạn hoán vị nhỏ.

## 15. Biến thể của bài

1. **LC 3376 — bản I:** hệ số tăng thêm `K` sau mỗi lần phá, `n ≤ 8` ⇒ DP bitmask.
2. Hệ số tăng `K` với `n = 80` ⇒ vẫn Hungary với `C[i][j] = ⌈s_i / (1 + jK)⌉`.
3. Phân công với ma trận chữ nhật (nhiều vị trí hơn khóa) ⇒ Hungary vẫn chạy (thêm cột).
4. Bài liên quan: LC 1066 (Campus Bikes II), LC 1947 (Maximum Compatibility Score Sum).

In [18]:
# Biến thể 1/2: hệ số tăng K sau mỗi lần phá
def min_time_K(strength, K):
    n = len(strength)
    return hungarian([[(strength[i] + (1 + j * K) - 1) // (1 + j * K) for j in range(n)] for i in range(n)])[0]

def brute_K(strength, K):
    best = None
    for p in itertools.permutations(range(len(strength))):
        t, X = 0, 1
        for i in p:
            t += (strength[i] + X - 1) // X; X += K
        best = t if best is None or t < best else best
    return best

assert min_time_K([3, 4, 1], 1) == 4 and min_time_K([2, 5, 4], 2) == 5           # ví dụ LC 3376
rng = random.Random(15)
for _ in range(200):
    a = [rng.randint(1, 50) for _ in range(rng.randint(1, 6))]; K = rng.randint(1, 5)
    assert min_time_K(a, K) == brute_K(a, K)
print('Biến thể hệ số tăng K khớp vét cạn (gồm ví dụ LC 3376: 4, 5).')

Biến thể hệ số tăng K khớp vét cạn (gồm ví dụ LC 3376: 4, 5).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi hoán vị | `O(n!)` | Oracle |
| 2 | DP bitmask | `O(2^n n)` | Bản I |
| 3 | Hungary | `O(n³)` | **Nộp bài** |
| 4 | Min-cost flow | chậm hơn | Tổng quát, như doocs |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[7]`.

**Bài 2 (tính tay):** `[4, 4]`.

**Bài 3 (điền TODO):** viết `best_order(strength)` trả về **thứ tự phá** tối ưu.

**Bài 4 (tự viết lại):** Hungary cho bài **cực đại** (đổi dấu chi phí).

**Bài 5 (mini challenge):** biến thể 3 — có `m ≥ n` "khe thời gian", khe `j` có hệ số `X_j` cho trước (không cần tăng dần); gán mỗi khóa vào một khe khác nhau.

In [19]:
def best_order_todo(strength):
    # TODO
    raise NotImplementedError

def hungarian_max_todo(W):
    # TODO
    raise NotImplementedError

def assign_to_slots_todo(strength, X):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một khóa, `X = 1`. (2) 7 phút. (3) `7`.

**Bài 2.** (1) `X = 1`: 4 phút; `X = 2`: 2 phút. (2) Tổng 6. (3) Thứ tự nào cũng như nhau.

**Bài 3.** (1) Hungary trả về phép gán khóa → vị trí. (2) Sắp khóa theo vị trí. (3) Mô phỏng để kiểm tra.

**Bài 4.** (1) `C' = −W`. (2) Giá trị tối ưu đổi dấu. (3) Đối chiếu vét cạn.

**Bài 5.** (1) Ma trận `n × m`, Hungary chữ nhật (thêm hàng giả chi phí 0 để vuông). (2) `C[i][j] = ⌈s_i / X_j⌉`. (3) Vét cạn kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().findMinimumTime([7]) == 7 and Solution().findMinimumTime([4, 4]) == 6

def best_order(strength):
    n = len(strength)
    total, assign = hungarian([[cost(strength[i], j) for j in range(n)] for i in range(n)])
    order = sorted(range(n), key=lambda i: assign[i])
    assert simulate(strength, order) == total
    return total, order

print('Bài 1: 7 | Bài 2: 6 | Bài 3 ví dụ 1:', best_order([3, 4, 1]), '| ví dụ 2:', best_order([2, 5, 4]))

Bài 1: 7 | Bài 2: 6 | Bài 3 ví dụ 1: (4, [2, 1, 0]) | ví dụ 2: (6, [0, 2, 1])


In [21]:
# Bài 4, 5
def hungarian_max(W):
    val, assign = hungarian([[-w for w in row] for row in W])
    return -val, assign

def assign_to_slots(strength, X):
    n, m = len(strength), len(X)
    C = [[(strength[i] + X[j] - 1) // X[j] for j in range(m)] for i in range(n)] + [[0] * m for _ in range(m - n)]
    val, assign = hungarian(C)
    return val

rng = random.Random(19)
for _ in range(200):
    n = rng.randint(1, 5)
    W = [[rng.randint(0, 9) for _ in range(n)] for _ in range(n)]
    assert hungarian_max(W)[0] == max(sum(W[i][p[i]] for i in range(n)) for p in itertools.permutations(range(n)))
    m = rng.randint(n, n + 2)
    a = [rng.randint(1, 30) for _ in range(n)]; X = [rng.randint(1, 6) for _ in range(m)]
    exp = min(sum((a[i] + X[p[i]] - 1) // X[p[i]] for i in range(n)) for p in itertools.permutations(range(m), n))
    assert assign_to_slots(a, X) == exp
print('Bài 4, 5 khớp vét cạn.')

Bài 4, 5 khớp vét cạn.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
C[i][j] = (s_i + j) // (j + 1)          # ⌈s_i/(j+1)⌉
Hungary (u, v, p, way):
  for i in 1..n: p[0] = i; tìm cột tự do qua minv/used, cập nhật thế vị bằng delta, đảo đường tăng
ans = -v[0]
```

**Checklist:**

- [ ] Làm tròn **lên** (`⌈s/X⌉`)?
- [ ] Hệ số vị trí `j` là `j + 1` (năng lượng về 0 sau mỗi lần phá)?
- [ ] Không dùng tham lam sắp xếp?
- [ ] `n = 80` ⇒ `O(n³)` (không bitmask)?
- [ ] Đối chiếu chữ ký nếu có Premium?